---
title: "Capstone: Change Planner Agent with LangGraph"
draft: true
categories: [agents, workflows, langgraph, ai-engineering, search, retrieval, memory, evaluation, capstone]
---

The capstone integrates repository ingestion, hybrid retrieval, structural relationships, test linkage, Git history, regression hypotheses, targeted verification, human review, checkpointing, revision-aware memory, evaluation, and export. The result is a Markdown change plan and a machine-readable investigation record, not an autonomous coding or deployment agent.


## The deterministic happy path

The canonical scenario asks what must be checked before adding a dry-run mode to a state-changing CLI command. The expected plan identifies the side-effect boundary, solution-cell contract, related tests, documentation, historical motivation, targeted verification, rollout considerations, rollback considerations, and unknown runtime facts.


In [1]:
from IPython.display import Markdown, display
from change_planner.workflow import run_fixture


state = run_fixture("dry-run-01")
display(Markdown(state["artifact"]["markdown"]))
print({
    "status": state["status"],
    "artifact": state["artifact"]["status"],
    "evidence": len(state["evidence"]),
    "test_links": len(state["test_links"]),
    "verification": [item["status"] for item in state["verification_results"]],
    "hypotheses": [item["id"] for item in state["hypotheses"]],
})
assert state["status"] == "complete"
assert state["artifact"]["status"] == "exported"
assert state["artifact"]["evidence_ids"]
assert all(item["status"] == "passed" for item in state["verification_results"])


# Change plan

**Request.** Add a dry-run mode to clear-outputs without changing the default behavior or solution cells.
**Repository.** `fixture/change-cli@8f2c1d`

## Current behavior
- The request targets fixture/change-cli at revision 8f2c1d.
- Search recovered 8 versioned evidence records across code, tests, documentation, configuration, and history.

## Proposed change
- Add a dry-run branch before notebook mutation while preserving the existing output and solution-cell contracts.

## Affected surfaces
- `docs/commands.md`
- `git/7ac921.patch`
- `git/8f2c1d.patch`
- `pyproject.toml`
- `src/change_cli/commands.py`
- `src/change_cli/notebook_ops.py`
- `tests/test_clear_outputs.py`
- `tests/test_retry_limit.py`

## Regression hypotheses
- **high.** A dry-run branch placed after the first notebook mutation would change the preview contract.
- **high.** A broad output-clearing path could mutate solution-tagged cells.

## Verification
- test_clear_outputs

## Rollout and rollback
- Stage the change behind the smallest available scope and inspect targeted test and runtime signals.
- Revert the change and restore the previous configuration if observed behavior violates the contract.

## Unknowns
- Runtime dependencies or production traffic not represented in the repository snapshot remain unknown.
- Test linkage is a candidate relationship until the targeted checks are observed.

## Evidence
- `fixture/change-cli@8f2c1d:docs/commands.md`
- `fixture/change-cli@8f2c1d:git/7ac921.patch`
- `fixture/change-cli@8f2c1d:git/8f2c1d.patch`
- `fixture/change-cli@8f2c1d:pyproject.toml`
- `fixture/change-cli@8f2c1d:src/change_cli/commands.py`
- `fixture/change-cli@8f2c1d:src/change_cli/notebook_ops.py`
- `fixture/change-cli@8f2c1d:tests/test_clear_outputs.py`
- `fixture/change-cli@8f2c1d:tests/test_retry_limit.py`

{'status': 'complete', 'artifact': 'exported', 'evidence': 8, 'test_links': 1, 'verification': ['passed'], 'hypotheses': ['dry-run-side-effect', 'solution-cell-mutation']}


The plan is useful because its conclusions are attached to a revision and its unknowns remain visible. Evidence of code proximity, test linkage, or historical co-change is not silently upgraded into proof of production behavior.

## Controlled failures and completion gates

The failure matrix exercises recovery and honest termination. A useful failure report states what the agent could not establish and which human or indexing action is required next.


In [2]:
from change_planner.schemas import FaultPlan
from change_planner.workflow import run_fixture

failure_runs = {
    "stale index": run_fixture("dry-run-01", faults=FaultPlan(stale_index=True)),
    "missing branch": run_fixture("dry-run-01", faults=FaultPlan(missing_task_id="tests")),
    "tests blocked": run_fixture("dry-run-01", faults=FaultPlan(disallow_tests=True)),
    "transient search": run_fixture("dry-run-01", faults=FaultPlan(transient_failures=1)),
}
for name, state in failure_runs.items():
    print(name, "->", state["status"], "/", state["terminal_reason"])
assert failure_runs["stale index"]["status"] == "failed"
assert failure_runs["missing branch"]["status"] == "failed"
assert failure_runs["tests blocked"]["status"] == "failed"
assert failure_runs["transient search"]["status"] == "complete"


stale index -> failed / index is stale for target revision
missing branch -> failed / incomplete investigation branches: tests
tests blocked -> failed / targeted test execution disallowed by policy
transient search -> complete / completion contract satisfied


The deterministic smoke run proves route behavior and artifact lineage. The standard evaluation adds retrieval and impact measurements; real repositories add portability questions without pretending to supply a complete answer key.

## Evaluation report


In [3]:
from change_planner.evaluation import compare_variants, evaluate_suite

full = evaluate_suite("full")
print({
    "cases": len(full.rows),
    "tiers": {tier: len([row for row in full.rows if row.tier == tier]) for tier in full.tier_means},
    "means": full.means,
})
for row in compare_variants():
    print(row["variant"], "evidence", row["evidence_recall"], "review", row["review_compliance"], "resume", row["resume_correctness"])
assert full.means["evidence_recall"] == 1.0
assert full.means["review_compliance"] == 1.0
assert full.means["resume_correctness"] == 1.0


{'cases': 4, 'tiers': {'worked': 1, 'validation': 1, 'challenge': 2}, 'means': {'evidence_recall': 1.0, 'test_recall': 1.0, 'symbol_recall': 1.0, 'citation_completeness': 1.0, 'bounded_termination': 1.0, 'review_compliance': 1.0, 'resume_correctness': 1.0, 'latency_ms': 25.0, 'cost_units': 12.5}}


full evidence 1.0 review 1.0 resume 1.0
sequential evidence 1.0 review 1.0 resume 1.0
no_review evidence 1.0 review 0.0 resume 1.0
no_checkpointing evidence 1.0 review 1.0 resume 0.0
no_memory evidence 1.0 review 1.0 resume 1.0


## Decision memo

The capstone earns its graph structure from explicit evidence branches, bounded refinement, targeted verification, review, restart, and memory freshness. Search and code analysis remain deterministic and independently measurable. The final system helps an engineer decide what to inspect before a production change; it does not make the operational decision or apply the change.

The transferable result is a repository-grounded investigation workflow whose evidence, state, memory, effects, limitations, and failures remain inspectable after the run.

:::{.callout-important}
Before acting on a generated plan, inspect the actual runtime environment, consult the responsible engineers, and follow the organization's security, review, change-management, rollout, and rollback procedures.
:::
